In [ ]:
import cv2
import numpy as np
import time
import os
import HandTrackingModule as htm

brushThickness = 15
detector = htm.handDetector(num_hands=1)
xp, yp = 0, 0
imgCanvas = np.zeros((720,1280,3), np.uint8)

cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)
try: 
    while True:
        success, img = cap.read()
        if not success:
            break
        img = cv2.flip(img,1)
    
        # 2. Find Landmarks
        img = detector.findHands(img)
        lmList = detector.findPosition(img, draw=False)
    
        if len(lmList) != 0:
            # tip of index and middle finger
            x1, y1 = lmList[8][1:]
            x2, y2 = lmList[12][1:]
    
            # 3. Check which fingures are up
            fingers = detector.fingersUp()
            # print(fingers)
    
            # 4. drawing 
            # if fingers[1] and fingers[2] == 0:
            if fingers == [0,1,0,0,0]:
                # xp, yp = 0, 0
                drawColor = (255,0,255)
                cv2.circle(img, (x1,y1),15,drawColor, cv2.FILLED)
                # print("Drawing mode")
                # draw based on points
                if xp==0 and yp==0:
                    xp, yp = x1,y1;
                cv2.line(img, (xp,yp), (x1,y1), drawColor, brushThickness)
                cv2.line(imgCanvas, (xp,yp), (x1,y1), drawColor, brushThickness)
                xp,yp=x1,y1
            else:
                xp, yp = 0, 0
                
            # 5 erasing mode
            # if fingers[0] and fingers[1] and fingers[2] and fingers[3] and fingers[4]:
            if fingers == [1,1,1,1,1]:
                drawColor = (0,0,0)
                x1, y1 = lmList[9][1:]
                x2, y2 = lmList[13][1:]
                cv2.circle(img, (x1,y1),0,drawColor)
                cv2.circle(imgCanvas, (x1,y1),150,drawColor, cv2.FILLED)
              # print("Erasing mode")

        imgGray = cv2.cvtColor(imgCanvas, cv2.COLOR_BGR2GRAY)
        _, imgInv = cv2.threshold(imgGray, 50, 255, cv2.THRESH_BINARY_INV)
        imgInv = cv2.cvtColor(imgInv, cv2.COLOR_GRAY2BGR)
        img = cv2.bitwise_and(img, imgInv) # will show in black on our img
        img = cv2.bitwise_or(img, imgCanvas) # will add color

        # img = cv2.addWeighted(img, 0.5, imgCanvas, 0.5, 0)
        cv2.imshow("Image", img)
        cv2.imshow("Canvas", imgCanvas)
        cv2.imshow("imgInv", imgInv)
        
        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

In [2]:
# folderPath = "Header"
# myList = os.listdir(folderPath)
# path(myList)